In [1]:
# ==========================================
# CELL 1: SETUP FILE PATHS
# ==========================================
import os
import pandas as pd

# Sesuaikan ekstensi file-nya (asumsi .csv)
path_kotor = "../../data/dataset_tiket_lengkap_lama_2.csv"
path_bersih = "../../data/dataset_tiket_lengkap_revisi.csv"

print(f"📥 Membaca data kotor dari: {path_kotor}")
print(f"📤 Data yang selamat akan disimpan ke: {path_bersih}")

# Cek apakah file kotornya benar-benar ada
if not os.path.exists(path_kotor):
    print("❌ ERROR: File lama tidak ditemukan! Pastikan nama dan foldernya benar.")
else:
    print("✅ File lama ditemukan. Siap dieksekusi!")

📥 Membaca data kotor dari: ../../data/dataset_tiket_lengkap_lama_2.csv
📤 Data yang selamat akan disimpan ke: ../../data/dataset_tiket_lengkap_revisi.csv
✅ File lama ditemukan. Siap dieksekusi!


In [2]:
# ==========================================
# CELL 2: SMART PARSER DENGAN 11 KOLOM KETAT
# ==========================================
import pandas as pd

print("🚑 Memulai Operasi Bedah Teks (Strict 11 Columns)...")

# Ini adalah struktur harga mati yang Anda minta
kolom_wajib = [
    "teks_keluhan_awam", "teks_laporan_teknisi", "tipe_aset", "lokasi_gedung", 
    "lokasi_lantai", "lokasi_zona", "kategori_aset", "severity", 
    "root_cause", "jenis_kerusakan", "biaya_perbaikan"
]

data_selamat = []
baris_gagal = 0
baris_total = 0

with open(path_kotor, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for line in lines:
    baris_total += 1
    
    # Pisahkan berdasarkan pipa dan bersihkan spasi
    parts = [p.strip() for p in line.split('|') if p.strip() != ""]
    
    if len(parts) < 11 or "teks_keluhan_awam" in line.lower():
        continue

    # PENDETEKSI TEKS AI (Keluhan & Laporan)
    teks_panjang = [p for p in parts if len(p) > 30]
    
    # Syarat mutlak: LLaMA harus nulis keluhan DAN laporan
    if len(teks_panjang) >= 2:
        keluhan = teks_panjang[0]
        laporan = teks_panjang[1]
        
        # Ekstrak 9 Metadata. 
        # Kita buang keluhan & laporan dari list parts agar murni sisa metadata
        metadata_raw = [p for p in parts if p not in teks_panjang]
        
        # Buang duplikat metadata (jika AI copas dobel seperti kasus Jet Shower)
        # Tapi tetap jaga urutan aslinya
        seen = set()
        metadata_unik = []
        for m in metadata_raw:
            if m not in seen:
                seen.add(m)
                metadata_unik.append(m)
                
        # Jika metadatanya lengkap ada 9 item (Tipe s.d Biaya)
        if len(metadata_unik) >= 9:
            data_selamat.append({
                "teks_keluhan_awam": keluhan,
                "teks_laporan_teknisi": laporan,
                "tipe_aset": metadata_unik[0],
                "lokasi_gedung": metadata_unik[1],
                "lokasi_lantai": metadata_unik[2],
                "lokasi_zona": metadata_unik[3],
                "kategori_aset": metadata_unik[4],
                "severity": metadata_unik[5], # Seharusnya ini berisi Ringan/Sedang/Berat/Fatal
                "root_cause": metadata_unik[6],
                "jenis_kerusakan": metadata_unik[7],
                "biaya_perbaikan": metadata_unik[8]
            })
        else:
            baris_gagal += 1
    else:
        # Jatuh ke sini jika AI tidak menulis keluhan (hanya laporan teknisi)
        baris_gagal += 1

print(f"🔍 Total Baris Diproses: {baris_total}")
print(f"✅ BERHASIL DISELAMATKAN: {len(data_selamat)} baris")
print(f"❌ DIBUANG (AI tidak membuat format 11 kolom): {baris_gagal} baris")

🚑 Memulai Operasi Bedah Teks (Strict 11 Columns)...
🔍 Total Baris Diproses: 649
✅ BERHASIL DISELAMATKAN: 458 baris
❌ DIBUANG (AI tidak membuat format 11 kolom): 160 baris


In [3]:
# ==========================================
# CELL 3: PREVIEW & SIMPAN HASIL REVISI (11 KOLOM)
# ==========================================

if len(data_selamat) > 0:
    # Jadikan DataFrame dengan urutan kolom yang Anda minta persis
    df_revisi = pd.DataFrame(data_selamat, columns=kolom_wajib)
    
    # PEMBERSIHAN FINAL: 
    # Pastikan kolom Severity benar-benar hanya berisi 4 kelas (buang yang salah geser)
    kelas_valid = ['Ringan', 'Sedang', 'Berat', 'Fatal']
    df_revisi = df_revisi[df_revisi['severity'].isin(kelas_valid)]
    
    # Hapus keluhan yang di-copas kembar identik oleh AI
    df_revisi = df_revisi.drop_duplicates(subset=['teks_keluhan_awam', 'teks_laporan_teknisi'])
    
    print(f"🏆 DATA FINAL YANG BERSIH & SELAMAT: {len(df_revisi)} baris!")
    
    # Simpan ke file revisi
    df_revisi.to_csv(path_bersih, sep='|', index=False)
    print(f"💾 Selesai! Data siap dipakai, disimpan di: {path_bersih}\n")
    
    display(df_revisi.head())
else:
    print("☠️ Operasi gagal. Tidak ada satupun data yang bisa diselamatkan ke format 11 kolom.")

🏆 DATA FINAL YANG BERSIH & SELAMAT: 229 baris!
💾 Selesai! Data siap dipakai, disimpan di: ../../data/dataset_tiket_lengkap_revisi.csv



,teks_keluhan_awam,teks_laporan_teknisi,tipe_aset,lokasi_gedung,lokasi_lantai,lokasi_zona,kategori_aset,severity,root_cause,jenis_kerusakan,biaya_perbaikan
0,AC Split di Gedung D Lantai 3 Zona Barat retak...,Telah dilakukan perbaikan pada AC Split di Ged...,AC Split,Gedung D,Lantai 3,Zona Barat,Mechanical,Fatal,Kelembaban tinggi,Retak/pecah,18771000
1,AC Split di Gedung D Lantai 3 Zona Barat pecah...,Telah dilakukan perbaikan pada AC Split di Ged...,AC Split,Gedung D,Lantai 3,Zona Barat,Mechanical,Fatal,Kelembaban tinggi,Retak/pecah,18771000
2,AC Split di Gedung D Lantai 3 Zona Barat menga...,Telah dilakukan perbaikan pada AC Split di Ged...,AC Split,Gedung D,Lantai 3,Zona Barat,Mechanical,Fatal,Kelembaban tinggi,Retak/pecah,18771000
3,Sistem access control di parkir lantai 3 agak ...,Teknisi telah memperbaiki masalah overheat pad...,Access Control,Gedung Parkir,Lantai 3,Zona Tengah,Security Sistem,Ringan,Human error,Overheat,356000
4,Sistem keamanan di lantai 3 zona tengah mengal...,Teknisi telah memperbaiki masalah overheat pad...,Access Control,Gedung Parkir,Lantai 3,Zona Tengah,Security Sistem,Ringan,Human error,Overheat,356000
